# 09B 根据尝试给提示，保存后继续

现在我们有一次真实执行结果。下一步不是把完整答案塞给学生，而是帮助他发现一个可修改的条件。学习状态来自程序记录，模型负责解释和提问。

本实验使用演示别名，不包含完整账号系统或多人服务的鉴权。

**本实践的新增能力**：不改变第八节的资料与导读基线，只把 09A 的失败证据接入分层提示和持久学习状态。分析时要同时对照确定性程序汇总与模型文字，特别核对“实际 2 项失败、4/6 通过”是否被模型说错。

**贯穿案例**

另一道到期卡片练习暴露 `due == today` 的边界错误。`LearningStore.next_hint` 让同一请求重试仍得到同一级提示，新请求才推进；修改后重跑测试并跨进程读取旧失败与新结果。程序生成的实测摘要与模型建议分开保存，防止流畅反馈掩盖测试计数错误。


## 学习路线与配套课件

本 Notebook 可以独立从头运行。先读定义和输入，预测结果，再执行并解释差异。小算例帮助理解机制；真实训练、工具执行与保存的 API 记录会分别标明来源。

本节重点：同一个提示请求重试，为什么不该升到下一级；用严格类型审计真实反馈背后的执行证据。

课件页：L09.02-S01、L09.03-S02、L09.05-S01、L09.05-S03。

沿原有课程单元顺序完成实验；新增对照放在相关代码后。练习先自行作答，展开参考分析后再比较。

In [1]:
from pathlib import Path
import sys, json, tempfile, subprocess, inspect, hashlib
from datetime import datetime, timezone
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
PROJECT = ROOT / "09 综合实践二 练习反馈与跨仓库迁移/代码/学习反馈"
SOURCE_PROJECT = ROOT / "08 综合实践一 资料与仓库导读/代码/学习助手"
sys.path.insert(0,str(SOURCE_PROJECT/"src"))
sys.path.insert(0,str(PROJECT/"src"))
from exercise_lab import LearningStore,run_exercise,catalog,CATALOG,verified_feedback
from corpus import Corpus
from tutor_runtime import TutorRuntime
from IPython.display import display,Code
print("默认离线；只执行课程自编的小练习，不执行资料仓库中的任意代码。")


默认离线；只执行课程自编的小练习，不执行资料仓库中的任意代码。


## 1. 分层提示与提前给答案

第一级指出观察位置；第二级帮助拆步骤；第三级更具体地描述方法。一次学生请求只取一级，而不是把三层拼成一个长答案。

下面先展开三层提示的设计，再观察运行时为什么只返回当前一级。尚未尝试、执行出错、用例失败和测试通过，需要不同的下一步。


In [2]:
for i,hint in enumerate(catalog()["due-cards"]["hints"],1):print(i,hint)
print("未尝试→先尝试；执行错误→先排查；用例失败→针对条件提示；测试通过→解释与新输入自检")


1 看失败用例中的 due 是否恰好等于 today，这张卡应不应该被选中？
2 把昨天、今天、明天三种边界分别列出来，先检查筛选条件。
3 到期包含今天；筛选后按日期升序，最后限制条数，不要把条数不变误当作限制失效。
未尝试→先尝试；执行错误→先排查；用例失败→针对条件提示；测试通过→解释与新输入自检


In [3]:
display(Code(inspect.getsource(LearningStore),language="python"))


class LearningStore:
    """仅本机教学用；learner_id 是演示别名，不是已实现完整账号鉴权。"""
    def __init__(self,path):
        path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
        self.db=sqlite3.connect(path)
        self.db.executescript('''CREATE TABLE IF NOT EXISTS lessons(
            id TEXT PRIMARY KEY, learner TEXT, corpus_version TEXT, exercise TEXT, hint_level INTEGER, status TEXT);
            CREATE TABLE IF NOT EXISTS attempts(id TEXT PRIMARY KEY,lesson_id TEXT,body TEXT);
            CREATE TABLE IF NOT EXISTS hint_receipts(lesson_id TEXT,request_id TEXT,body TEXT,PRIMARY KEY(lesson_id,request_id));''')
    def create(self,learner,corpus_version,exercise_id):
        if not re.fullmatch(r'[a-zA-Z0-9_-]{1,40}',learner):raise ValueError('学习者使用简短演示别名')
        if exercise_id not in catalog():raise ValueError('未知练习')
        lesson_id=uuid.uuid4().hex
        with self.db:self.db.execute('INSERT INTO lessons VALUES(?,?,?,?,?,?)',
            (lesson_id,learner,corpus_version,exercise_id,0,'not_attempted'))
        return lesson_id
    def read(self,lesson_id,learner,corpus_version):
        row=self.db.execute('SELECT * FROM lessons WHERE id=? AND learner=?',(lesson_id,learner)).fetchone()
        if row is None:raise KeyError('学习记录不存在或别名不匹配')
        if row[2]!=corpus_version:raise ValueError('资料版本变化，先重新核查练习')
        attempts=[json.loads(r[0]) for r in self.db.execute('SELECT body FROM attempts WHERE lesson_id=? ORDER BY rowid',(lesson_id,))]
        return {'lesson_id':row[0],'learner':row[1],'corpus_version':row[2],'exercise_id':row[3],
                'hint_level':row[4],'status':row[5],'attempts':attempts}
    def add_attempt(self,lesson_id,learner,corpus_version,attempt):
        state=self.read(lesson_id,learner,corpus_version)
        if attempt['exercise_id']!=state['exercise_id']:raise ValueError('练习不匹配')
        if attempt['catalog_sha256']!=hashlib.sha256(CATALOG.read_bytes()).hexdigest():raise ValueError('题目版本变化')
        status='tests_passed' if attempt['result']['passed'] else 'needs_work'
        if attempt['result'].get('execution_error'):status='execution_error'
        with self.db:
            self.db.execute('INSERT INTO attempts VALUES(?,?,?)',(attempt['attempt_id'],lesson_id,json.dumps(attempt,ensure_ascii=False)))
            self.db.execute('UPDATE lessons SET status=? WHERE id=?',(status,lesson_id))
        return self.read(lesson_id,learner,corpus_version)
    def next_hint(self,lesson_id,learner,corpus_version,request_id):
        state=self.read(lesson_id,learner,corpus_version)
        if state['status']=='not_attempted':return {'level':0,'hint':'先写一次预测或尝试，再选择需要帮助的地方。'}
        if state['status']=='tests_passed':return {'level':state['hint_level'],'hint':'随课测试已通过。请解释边界条件，再用一组新输入自检；这还不是掌握证明。'}
        if state['status']=='execution_error':return {'level':0,'hint':'先检查运行错误或超时，再判断算法结果；现在不能说测试全部通过。'}
        if not isinstance(request_id,str) or not re.fullmatch(r'[a-zA-Z0-9_-]{1,80}',request_id):raise ValueError('提示请求标识不合法')
        # 同一个显式请求即使重复执行，也不能越过一级提示；事务内检查并写回。
        with self.db:
            self.db.execute('BEGIN IMMEDIATE')
            receipt=self.db.execute('SELECT body FROM hint_receipts WHERE lesson_id=? AND request_id=?',(lesson_id,request_id)).fetchone()
            if receipt:return json.loads(receipt[0])
            current=self.db.execute('SELECT hint_level FROM lessons WHERE id=?',(lesson_id,)).fetchone()[0]
            hints=catalog()[state['exercise_id']]['hints'];level=min(current+1,len(hints))
            result={'level':level,'hint':hints[level-1],'scope':'只提供当前一级提示，不返回完整参考实现'}
            self.db.execute('UPDATE lessons SET hint_level=? WHERE id=?',(level,lesson_id))
            self.db.execute('INSERT INTO hint_receipts VALUES(?,?,?)',(lesson_id,request_id,json.dumps(result,ensure_ascii=False)))
        return result
    def close(self):self.db.close()

In [4]:
workspace=Path(tempfile.mkdtemp(prefix="course-feedback-"))
version=Corpus(SOURCE_PROJECT/"fixtures/flashcards").fingerprint
store=LearningStore(workspace/"learning.sqlite3")
lesson=store.create("student-a",version,"due-cards")
failed=run_exercise("due-cards",PROJECT/"exercises/due-cards/student.py")
state=store.add_attempt(lesson,"student-a",version,failed)
print(state["status"],len(state["attempts"]))
first=store.next_hint(lesson,"student-a",version,"request-1")
replayed=store.next_hint(lesson,"student-a",version,"request-1")
print(first)
assert first==replayed and first["level"]==1


needs_work 1


{'level': 1, 'hint': '看失败用例中的 due 是否恰好等于 today，这张卡应不应该被选中？', 'scope': '只提供当前一级提示，不返回完整参考实现'}


### 同一个提示请求重试，为什么不该升到下一级

配套课件：L09.02-S01、L09.05-S01。

网络或界面重试可能重复提交同一请求。LearningStore 用请求 ID 关联这次提示；重复 ID 应得到同一结果，新 ID 才推进一级。下面沿用刚才的学习记录，观察两个状态变化。

In [5]:
case_same = store.next_hint(lesson, "student-a", version, "request-1")
case_next = store.next_hint(lesson, "student-a", version, "request-2")
case_next_again = store.next_hint(lesson, "student-a", version, "request-2")
print("同一请求再次返回:", case_same["level"])
print("新请求返回:", case_next["level"])
print("新请求重试:", case_next_again["level"])
assert case_same == first and case_same["level"] == 1
assert case_next == case_next_again and case_next["level"] == 2


同一请求再次返回: 1
新请求返回: 2
新请求重试: 2


**结果解读**

结果依次是 1、2、2。重试一次请求不会意外消耗下一级提示；新请求才推进学习状态。这里修改的是本机新建练习记录，后面的真实模型记录是另一份历史证据。

**小练习**

若前端每次重试都生成新 ID，会发生什么？稳定 ID 应在什么时候生成？

<details><summary>完成后展开参考分析</summary>

重试会被误认为新的提示需求，连续升级。应在一次逻辑请求开始时生成 ID，在该请求所有重试中复用。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 2. 学习状态接入同一个 Harness

TutorRuntime 在导读环境上增加两个工具：read_learning_state 读最新尝试，request_next_hint 取当前提示。工具参数不能任意选择学生或执行文件；这些范围由宿主绑定。

同一个提示请求有固定 ID。若恢复过程中重放，不会悄悄从一级推进到二级。新的学生提交会改变运行快照，旧模型任务不能继续假装仍在评价原提交。


In [6]:
display(Code(inspect.getsource(TutorRuntime),language="python"))
runtime=TutorRuntime(SOURCE_PROJECT/"fixtures/flashcards",store,lesson,"student-a","request-1")
print("工具:",sorted(runtime.allowed))
assert "run_code" not in runtime.allowed and "set_status" not in runtime.allowed


class TutorRuntime(StudyRuntime):
    allowed={'read_learning_state','request_next_hint'}
    def __init__(self,directory,learning_store,lesson_id,learner,hint_request=None):
        super().__init__(directory)
        self.learning,self.lesson_id,self.learner=learning_store,lesson_id,learner
        self.hint_request=hint_request
        self.learning_state()  # 创建环境时先核对学生与资料版本。
        self.identity+=':lesson:'+lesson_id+':learner:'+learner+':hint:'+str(hint_request)
    def learning_state(self):
        return self.learning.read(self.lesson_id,self.learner,self.corpus.fingerprint)
    def snapshot(self):
        result=super().snapshot();state=self.learning_state()
        result['exercise/catalog']=hashlib.sha256(CATALOG.read_bytes()).hexdigest()
        result['learning/attempts']=hashlib.sha256(json.dumps(state['attempts'],sort_keys=True).encode()).hexdigest()
        return result
    def instructions(self,library):
        state=self.learning_state();exercise=catalog()[state['exercise_id']]
        return ('你是面向大学初学者的练习反馈助手。当前阶段只处理宿主绑定的练习。'
            '先读取本次学习记录；只依据实际测试说明状态。'
            '测试通过只能说随课用例通过，不能宣称已经掌握。若有提示请求，最多调用一次 request_next_hint；'
            '给当前一级提示，不给参考实现，不替学生修改或执行代码。200字以内，让学生自己尝试。'
            '\n练习要求：'+exercise['prompt']+'\n本轮是否由学生明确请求提示：'+str(self.hint_request is not None))
    def declarations(self,tools):
        # 反馈阶段只暴露所需动作；导读阶段仍由 StudyRuntime 提供资料工具。
        return [
            declaration('read_learning_state','读取宿主绑定的当前练习、最新实测结果与提示级别。',{},[]),
            declaration('request_next_hint','学生明确请求后，取得当前一级提示；重复调用返回同一提示。',{},[])]
    async def execute(self,client,library,name,args,remote_names):
        if name in {'read_learning_state','request_next_hint'}:
            if args!={}:raise ValueError('学习工具不接受学生 ID、路径或代码参数')
            if name=='read_learning_state':
                state=self.learning_state()
                return {'ok':True,'data':{k:v for k,v in state.items() if k!='attempts'}|
                    {'latest_attempt':state['attempts'][-1] if state['attempts'] else None}}
            if self.hint_request is None:raise PermissionError('本轮没有学生提示请求')
            return {'ok':True,'data':self.learning.next_hint(self.lesson_id,self.learner,self.corpus.fingerprint,self.hint_request)}
        return await super().execute(client,library,name,args,remote_names)

工具: ['read_learning_state', 'request_next_hint']


## 3. 看真实模型如何使用测试结果

以下读取一份已经保存的真实模型调用记录。先打印记录时间和模型名称，再核对实际工具、提示是否越级、是否直接给出完整代码，以及是否错称测试全过。

模型名称只描述本次运行条件，不是未来重跑时的固定推荐；以后重跑，应先查询当时可用模型并记录实际选择。不要只用“读起来像老师”作为通过标准，对应尝试和返回提示必须能在同一记录中回查。


In [7]:
report=json.loads((PROJECT/"results/deepseek-feedback.json").read_text(encoding='utf-8'))
print("本次真实记录:",report.get("recorded_at"),"模型:",report.get("model","未记录"))
original=next(c for c in report["cases"] if c["exercise_id"]=="due-cards")
print("原始任务状态:",[p["state"]["status"] for p in original["phases"]])
focused=json.loads((PROJECT/"results/deepseek-feedback-focused.json").read_text(encoding='utf-8'))
case=focused["cases"][0]
phase=case["phases"][0]
print("实测状态:",phase["learning"]["status"],"提示级别:",phase["learning"]["hint_level"])
print("Agent 状态:",phase["state"]["status"],"工具次数:",phase["state"]["tool_calls"])
print(phase["state"]["answer"])
for artifact in phase["artifacts"]:
    data=artifact["body"].get("data",{})
    if isinstance(data,dict) and "hint" in data:print("实际返回的一级提示:",data["hint"])


本次真实记录: 2026-09-24T05:53:49.483320+00:00 模型: deepseek-flash
原始任务状态: ['final_answer', 'final_answer']
实测状态: needs_work 提示级别: 1
Agent 状态: final_answer 工具次数: 2
实测：date-order、limit-one、exclude-future、invalid-limit 通过；equal-today 和 stable-tie 未过，说明到期筛选或同天排序有偏差。

提示：检查 due 恰好等于 today 的卡片是否被选中了。同一天的多张卡要保持原顺序，排序时用稳定排序（如按 due 排序即可，别打乱同键顺序）。改完自己重跑验证。随课用例通过只代表提交测试通过，不等于已掌握。
实际返回的一级提示: 看失败用例中的 due 是否恰好等于 today，这张卡应不应该被选中？


### 用严格类型审计真实反馈背后的执行证据

配套课件：L09.03-S02、L09.05-S01、L09.05-S03。

pydantic 对真实反馈记录中的尝试快照做最小结构校验，严格区分布尔通过值与数字；再把未通过测试数和模型回答并列。结构合法仍不代表解释准确。

In [8]:
from pydantic import BaseModel, ConfigDict, StrictBool
class CaseTestEvidence(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: str
    passed: StrictBool

for case_item in report["cases"]:
    for case_phase in case_item["phases"]:
        latest = case_phase["learning"]["attempts"][-1]
        tests = [CaseTestEvidence.model_validate(row) for row in latest["result"]["tests"]]
        failed = [row.id for row in tests if not row.passed]
        print(case_item["exercise_id"], case_phase["phase"],
              "实际执行:", latest["execution"], "失败用例:", failed,
              "模型停止:", case_phase["state"]["status"])
        assert latest["execution"] == "actual_python_subprocess"
        assert latest["result"]["passed"] == (len(failed) == 0)
print("请对照每阶段原始回答逐条检查：是否与失败用例一致，是否只给当前级别提示？")


rank-positive needs_work 实际执行: actual_python_subprocess 失败用例: ['filter-zero', 'descending', 'invalid-limit', 'boolean-limit'] 模型停止: final_answer
rank-positive tests_passed 实际执行: actual_python_subprocess 失败用例: [] 模型停止: final_answer
due-cards needs_work 实际执行: actual_python_subprocess 失败用例: ['equal-today', 'stable-tie'] 模型停止: final_answer
due-cards tests_passed 实际执行: actual_python_subprocess 失败用例: [] 模型停止: final_answer
请对照每阶段原始回答逐条检查：是否与失败用例一致，是否只给当前级别提示？


**结果解读**

程序测试结果是练习事实，模型反馈是对事实的解释。结构校验能发现类型和汇总矛盾，不能替代阅读回答。真实学生的学习效果还需独立测量。

**小练习**

模型说“全部通过”而快照有失败用例时，应保留哪两份证据，下一步改哪里？

<details><summary>完成后展开参考分析</summary>

保留原始模型回答和执行快照，定位反馈生成/核验层；不能重写测试结果让回答看起来正确。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

### 回答流畅仍可能数错

收窄工具后的日期任务完成了，但模型把实际 **2 项失败**说成 4 项，还提前说出了应修改的比较符号。它同时把几个用例通过推成“排序与截断没问题”，这个范围过大。

因此学习者首先看到下面由确定性程序汇总的实测摘要和原始一级提示；模型补充文字单独核验，不作为完成依据。这个改动不声称已经自动判断所有教学质量问题。


In [9]:
display(Code(inspect.getsource(verified_feedback),language="python"))
hint=next(a["body"]["data"] for a in phase["artifacts"] if "hint" in a["body"].get("data",{}))
summary=verified_feedback(phase["learning"],hint)
print(summary)
assert len(summary["failed_test_ids"])==2 and summary["message"].startswith("随课用例 4/6 通过")


def verified_feedback(state,hint=None):
    """学生首先看到程序生成的实测摘要；模型补充建议另列为待核验内容。"""
    if not state['attempts']:return {'status':'not_attempted','message':'先完成一次尝试。'}
    result=state['attempts'][-1]['result'];tests=result['tests']
    if result.get('execution_error'):return {'status':'execution_error','message':'执行未完成，先排查错误。','error_type':result['execution_error']}
    failed=[t['id'] for t in tests if not t['passed']]
    message=f"随课用例 {len(tests)-len(failed)}/{len(tests)} 通过。"
    if not failed:message+='这不等于已经掌握；请解释一个边界并尝试新输入。'
    return {'status':state['status'],'message':message,'failed_test_ids':failed,
            'hint':hint.get('hint') if hint else None,'model_advice':'需对照上述实测记录核验'}

{'status': 'needs_work', 'message': '随课用例 4/6 通过。', 'failed_test_ids': ['equal-today', 'stable-tie'], 'hint': '看失败用例中的 due 是否恰好等于 today，这张卡应不应该被选中？', 'model_advice': '需对照上述实测记录核验'}


## 4. 提交修改后，退出并重新打开

这里显式选择课程提供的教学修订示例，模拟完成修改后的第二次提交；这不表示模型已经替学习者修改。两次尝试都保留，提示不会覆盖旧结果。

练习记录与 Agent 任务是两种状态：前者记学习过程，后者记某次模型与工具运行。换了一次学生提交，可以启动新反馈任务，同时继续同一条学习记录。


In [10]:
fixed=run_exercise("due-cards",PROJECT/"exercises/due-cards/teaching_revision.py")
store.add_attempt(lesson,"student-a",version,fixed)
store.close()
store=LearningStore(workspace/"learning.sqlite3")
continued=store.read(lesson,"student-a",version)
print("同一学习记录:",continued["lesson_id"]==lesson)
print("历史尝试:",[a["result"]["passed"] for a in continued["attempts"]])
print("当前状态:",continued["status"])
assert len(continued["attempts"])==2 and continued["status"]=="tests_passed"


同一学习记录: True
历史尝试: [False, True]
当前状态: tests_passed


## 练习：换了学习者或资料版本会怎样

预测两个错误读取会得到什么结果。说明为什么只存一段“这个学生学到排序了”的自然语言摘要，还不足以定位本次尝试。


In [11]:
for learner,source_version in [("student-b",version),("student-a","changed-version")]:
    try:store.read(lesson,learner,source_version)
    except (KeyError,ValueError) as error:print(type(error).__name__,str(error))


KeyError '学习记录不存在或别名不匹配'
ValueError 资料版本变化，先重新核查练习


In [12]:
assert continued["learner"]=="student-a" and continued["corpus_version"]==version
print("学习者、资料版本、题目与尝试结果共同说明状态；演示别名仍不是完整身份认证。")
store.close()


学习者、资料版本、题目与尝试结果共同说明状态；演示别名仍不是完整身份认证。


## 5. 导出反馈、提示与双状态审计

一份可迁移的记录要同时说明：模型证据来自何时何种模型、程序证据是什么、提示请求如何幂等，以及学习状态与 Agent 运行状态为何不能混为一谈。


In [13]:
out=ROOT/".local/student-runs/l09";out.mkdir(parents=True,exist_ok=True)
model_evidence={"recorded_at":focused.get("recorded_at"),"model":focused.get("model","未记录"),
                "future_rerun_rule":"重跑前查询当时可用模型；保存实际 model、recorded_at 与原始响应，不把历史模型名当作长期默认值"}
audit={
    "artifact":"feedback-and-state-audit",
    "model_evidence":model_evidence,
    "program_evidence":summary,
    "hint_receipt":{"request_id":"request-1","level":first["level"],"replayed_identically":first==replayed},
    "identity":{"learner":continued["learner"],"corpus_version":continued["corpus_version"],"exercise_id":continued["exercise_id"]},
    "learning_state":{"lesson_id":continued["lesson_id"],"status":continued["status"],"attempt_count":len(continued["attempts"])},
    "agent_run_state":{"historical_run_id":phase["state"]["run_id"],"status":phase["state"]["status"],"tool_calls":phase["state"]["tool_calls"]},
    "boundary":"测试与提示推进学习记录；一次模型运行的完成状态不能替代学习状态"
}
json_path=out/"feedback-and-state-audit.json";md_path=out/"feedback-and-state-audit.md"
json_path.write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding="utf-8")
md_path.write_text("\n".join(["# 反馈与状态审计","",f"- 历史模型：{model_evidence['model']}",
    f"- 记录时间：{model_evidence['recorded_at']}",f"- 程序摘要：{summary['message']}",
    f"- 同请求幂等：{audit['hint_receipt']['replayed_identically']}",f"- 学习状态：{continued['status']}",
    f"- 状态边界：{audit['boundary']}",f"- 未来重跑：{model_evidence['future_rerun_rule']}"]),encoding="utf-8")
print(json_path);print(md_path)


.local\student-runs\l09\feedback-and-state-audit.json
.local\student-runs\l09\feedback-and-state-audit.md


## 回顾

一次请求一级提示；模型读取实测结果，不直接改完成状态；提示重放保持级别；重新打开后保留全部尝试；资料或学生不匹配时拒绝复用。
